# Lab 2 — Baseline com múltiplos datasets

Neste notebook vamos rodar o mesmo pipeline de baseline para 4 datasets diferentes.

A ideia é simples: para cada dataset na lista `keys_to_run`, o código vai:
1. baixar e carregar os dados
2. limpar valores ausentes e aplicar `dropna` antes do split
3. treinar dois modelos baseline (um linear e um de floresta)
4. imprimir as métricas e salvar os resultados

No final, uma tabela compara os melhores modelos de cada dataset.

In [9]:
import os

base = r"C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga"

for root, dirs, files in os.walk(base):
    print(root)
    for d in dirs:
        print("  [DIR]", d)
    for f in files:
        print("  [FILE]", f)

C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga
  [DIR] corruptions
  [DIR] evaluation
  [DIR] tasks
  [DIR] __pycache__
  [FILE] basis.py
  [FILE] utils.py
  [FILE] __init__.py
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\corruptions
  [DIR] __pycache__
  [FILE] generic.py
  [FILE] image.py
  [FILE] numerical.py
  [FILE] text.py
  [FILE] __init__.py
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\corruptions\__pycache__
  [FILE] generic.cpython-39.pyc
  [FILE] __init__.cpython-39.pyc
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\evaluation
  [FILE] basis.py
  [FILE] corruption_impact.py
  [FILE] schema_stresstest.py
  [FILE] schema_validation.py
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\tasks
  [FILE] income.py
  [FILE] openml.py
  [FILE] reviews.py
  [FILE] shoes.py
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\__pycache__
  [FILE] basis.cpython-39.pyc
  [FILE] utils.cpython-39.pyc
  [FILE] __init__.cpytho

In [10]:
# 1) Bibliotecas
from __future__ import annotations

import json
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings('ignore', category=ConvergenceWarning)

from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

import sys
from pathlib import Path

import sys
from pathlib import Path

BASE_DIR = Path().resolve()
JENGA_PATH = BASE_DIR / "jenga" / "src"

sys.path.insert(0, str(JENGA_PATH))

import jenga
print("Jenga path:", jenga.__file__)

from jenga.corruptions.generic import MissingValues
print("IMPORT OK")

from jenga.corruptions.generic import MissingValues

import inspect
print(inspect.signature(MissingValues))

from joblib import Parallel, delayed

from lightgbm import LGBMClassifier, LGBMRegressor

import miceforest as mf

# Suprimir warnings verbosos do LightGBM e sklearn em produção

import logging
warnings.filterwarnings('ignore', category=UserWarning, module='lightgbm')
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
warnings.filterwarnings('ignore', category=FutureWarning)
logging.getLogger('lightgbm').setLevel(logging.ERROR)


Jenga path: C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\__init__.py
IMPORT OK
(column, fraction, na_value=nan, missingness='MCAR')


In [11]:
# 2) Metadados dos datasets e funcoes  que nos ajudam a baixar, extrair e normalizar os dados

DATASETS = {
    "adult": {"task": "classification", "target": "income"},
    "bank_marketing": {"task": "classification", "target": "y"},
    "air_quality_uci": {"task": "regression", "target": "C6H6(GT)"},
    "communities_crime": {"task": "regression", "target": "ViolentCrimesPerPop"},
}

# Tokens que representam valores ausentes
NUMERIC_MISSING_VALUES = [-200, -200.0]

STRING_MISSING_VALUES = [
    "?", " ?", "? ", "NA", "N/A", "na", "n/a", "NaN", "nan", "", " ",
    "unknown", "Unknown", "-200",
]


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def download_file(url: str, output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not output_path.exists():
        print(f"[download] {url}")
        urlretrieve(url, output_path)
    else:
        print(f"[cache] {output_path}")
    return output_path


def unzip_file(zip_path: Path, extract_dir: Path) -> Path:
    extract_dir.mkdir(parents=True, exist_ok=True)
    marker = extract_dir / ".extracted"
    if not marker.exists():
        print(f"[extract] {zip_path}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
        marker.write_text("ok\n", encoding="utf-8")
    else:
        print(f"[cache] {extract_dir}")
    return extract_dir


def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Substitui tokens de missing por NaN, coluna a coluna, respeitando o dtype.
    
    - Colunas numericas: substitui -200 e -200.0
    - Colunas de texto: faz strip e substitui tokens como '?', 'unknown', etc.
    Essa abordagem evita erros do pandas 2.x ao misturar int e str no replace().
    -- TODO: verificar se tem outros simbolos estranhos nos dataset da atividade (pedir par aos alunos checarem isso)
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].replace(NUMERIC_MISSING_VALUES, np.nan)
        else:
            df[col] = df[col].astype("string").str.strip()
            df[col] = df[col].replace(STRING_MISSING_VALUES, pd.NA)
    return df
def to_native_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converte TODAS as colunas com dtypes de extensão do pandas para tipos nativos,
    eliminando pd.NA que causa 'boolean value of NA is ambiguous'.
    Cobre: StringDtype, BooleanDtype, Int8/16/32/64Dtype, Float32/64Dtype, etc.
    """
    df = df.copy()
    for col in df.columns:
        dtype = df[col].dtype
        # Detecta qualquer ExtensionDtype (StringDtype, BooleanDtype, Int64Dtype, etc.)
        if not isinstance(dtype, np.dtype):
            if hasattr(dtype, "numpy_dtype"):
                # Tipos numéricos nullable (Int64, Float64) → float64 nativo
                # (float64 aceita np.nan; int não aceita, por isso usa float)
                try:
                    df[col] = df[col].astype(object).where(
                        df[col].notna(), other=np.nan
                    ).astype(float)
                    continue
                except (ValueError, TypeError):
                    pass
            # Todos os outros (StringDtype, BooleanDtype, CategoricalDtype, etc.) → object
            df[col] = df[col].astype(object).where(df[col].notna(), other=None)
    return df

def _to_object_dtype(X):
    if hasattr(X, "astype"):
        return X.astype(object)
    return X

def _to_str_dtype(X):
    return X.astype(str)

In [12]:
# 3) Funcoes. para baixar os datasets automaticamente

# fiz alguns pequenos ajustes caso a caso para facilitar a leitura e evitar erros de parsing

def load_adult(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "adult"
    data_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
        base / "adult.data",
    )
    test_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
        base / "adult.test",
    )

    columns = [
        "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
        "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
        "hours_per_week", "native_country", "income",
    ]

    train_df = pd.read_csv(data_file, header=None, names=columns, skipinitialspace=True, na_values=["?", " ?"])
    test_df = pd.read_csv(test_file, header=None, names=columns, skiprows=1, skipinitialspace=True, na_values=["?", " ?"])

    df = pd.concat([train_df, test_df], ignore_index=True)
    df["income"] = df["income"].astype(str).str.replace(".", "", regex=False).str.strip()
    return df, "income", "classification"


def load_bank_marketing(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "bank_marketing"
    zip_path = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip",
        base / "bank.zip",
    )
    extract_dir = unzip_file(zip_path, base / "extracted")
    df = pd.read_csv(extract_dir / "bank-full.csv", sep=";")
    return df, "y", "classification"


def load_air_quality_uci(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "air_quality_uci"
    zip_path = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip",
        base / "AirQualityUCI.zip",
    )
    extract_dir = unzip_file(zip_path, base / "extracted")
    candidates = list(extract_dir.rglob("AirQualityUCI.csv"))
    if not candidates:
        raise FileNotFoundError("Nao encontrei AirQualityUCI.csv dentro do zip.")

    df = pd.read_csv(candidates[0], sep=";", decimal=",")
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    for col in ["Date", "Time"]:
        if col in df.columns:
            df = df.drop(columns=[col])
    return df, "C6H6(GT)", "regression"


def load_communities_crime(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "communities_crime"
    data_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data",
        base / "communities.data",
    )
    df = pd.read_csv(data_file, header=None, na_values=["?"])

    cols = [f"col_{i}" for i in range(df.shape[1])]
    cols[-1] = "ViolentCrimesPerPop"
    df.columns = cols
    # colunas  com muito problema
    df = df.drop(columns=[c for c in ["col_0", "col_1", "col_2", "col_3", "col_4"] if c in df.columns])
    return df, "ViolentCrimesPerPop", "regression"


def load_dataset(key: str, data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    if key == "adult":
        return load_adult(data_dir)
    if key == "bank_marketing":
        return load_bank_marketing(data_dir)
    if key == "air_quality_uci":
        return load_air_quality_uci(data_dir)
    if key == "communities_crime":
        return load_communities_crime(data_dir)
    raise ValueError(f"Key desconhecida: {key}")

def inject_mcar(df: pd.DataFrame, missing_rate: float, random_state: int = 42) -> pd.DataFrame:
    corruption = MissingValues(missing_rate=missing_rate, random_state=random_state)
    df_corrupted = corruption.transform(df.copy())
    return df_corrupted


def inject_mar(df: pd.DataFrame, missing_rate: float, random_state: int = 42) -> pd.DataFrame:
    df_corrupted = df.copy()

    np.random.seed(random_state)

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) == 0:
        return inject_mcar(df, missing_rate, random_state)

    ref_col = numeric_cols[0]

    threshold = df[ref_col].median()

    mask = df[ref_col] > threshold

    for col in df.columns:
        prob = np.random.rand(len(df))
        df_corrupted.loc[mask & (prob < missing_rate), col] = np.nan

    return df_corrupted


def inject_mnar(df: pd.DataFrame, missing_rate: float, random_state: int = 42) -> pd.DataFrame:
    df_corrupted = df.copy()

    np.random.seed(random_state)

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) == 0:
        return inject_mcar(df, missing_rate, random_state)

    for col in numeric_cols:
        threshold = df[col].median()
        mask = df[col] > threshold

        prob = np.random.rand(len(df))
        df_corrupted.loc[mask & (prob < missing_rate), col] = np.nan

    return df_corrupted

In [13]:
# 4) Funcoes para separar treino e teste

FAST_MODE            = False
MISSING_MECHANISMS   = ["MCAR", "MAR", "MNAR"]
MISSING_RATES        = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
N_ESTIMATORS         = 50   # Random Forest e modelos básicos
N_ESTIMATORS_LGBM    = 100  # LightGBM precisa de mais árvores (leaf-wise)
MAX_ITER_IMPUTER     = 5
SAMPLE_SIZE          = None

def split_X_y(df: pd.DataFrame, target: str) -> tuple[pd.DataFrame, pd.Series]:
    if target not in df.columns:
        raise ValueError(f"Target '{target}' nao esta no dataset.")
    return df.drop(columns=[target]), df[target]


def build_preprocessor(X: pd.DataFrame, scale_numeric: bool) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler() if scale_numeric else "passthrough"),
    ])

    cat_pipeline = Pipeline([
        ("to_object", FunctionTransformer(_to_object_dtype)),  # função nomeada
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("to_str", FunctionTransformer(_to_str_dtype)),        # função nomeada
        ("onehot", make_one_hot_encoder()),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", num_pipeline, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", cat_pipeline, categorical_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_preprocessor_with_imputer(
    X: pd.DataFrame,
    imputer_type: str,
    scale_numeric: bool
) -> ColumnTransformer:
    """
    imputer_type aceito:
      "dropna"        – sem imputação (linhas com NaN já foram removidas antes)
      "simple_mean"   – SimpleImputer(strategy="mean")
      "simple_median" – SimpleImputer(strategy="median")
      "simple_freq"   – SimpleImputer(strategy="most_frequent")
      "simple_const"  – SimpleImputer(strategy="constant", fill_value=0)
      "knn_3"         – KNNImputer(n_neighbors=3)
      "knn_5"         – KNNImputer(n_neighbors=5)
      "knn_7"         – KNNImputer(n_neighbors=7)
      "iterative"     – IterativeImputer
    """
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    # ── imputador numérico ──────────────────────────────────────────────────
    if imputer_type in ("dropna", "miceforest"):
        # dados já vieram sem NaN (dropna removeu linhas; miceforest já imputou)
        num_imputer = SimpleImputer(strategy="mean")
    elif imputer_type == "simple_mean":
        num_imputer = SimpleImputer(strategy="mean")
    elif imputer_type == "simple_median":
        num_imputer = SimpleImputer(strategy="median")
    elif imputer_type == "simple_freq":
        num_imputer = SimpleImputer(strategy="most_frequent")
    elif imputer_type == "simple_const":
        num_imputer = SimpleImputer(strategy="constant", fill_value=0)
    elif imputer_type.startswith("knn_"):
        k = int(imputer_type.split("_")[1])
        num_imputer = KNNImputer(n_neighbors=k)
    elif imputer_type == "iterative":
        num_imputer = IterativeImputer(
            max_iter=MAX_ITER_IMPUTER,
            initial_strategy="mean",
            random_state=0,
        )
    else:
        raise ValueError(f"Imputer desconhecido: {imputer_type}")

    num_pipeline = Pipeline([
        ("imputer", num_imputer),
        ("scaler", StandardScaler() if scale_numeric else "passthrough"),
    ])

    cat_pipeline = Pipeline([
        ("to_object", FunctionTransformer(_to_object_dtype)),
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("to_str", FunctionTransformer(_to_str_dtype)),
        ("onehot", make_one_hot_encoder()),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", num_pipeline, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", cat_pipeline, categorical_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def inject_missing_values(df, target, mechanism, missing_rate, random_state):
    df_corrupted = df.copy()

    y = df_corrupted[target]
    X = df_corrupted.drop(columns=[target]).copy()

    X_original = X.copy()

    for col in X.columns:
        corruption = MissingValues(
            column=col,
            fraction=missing_rate,
            missingness=mechanism
        )
        X = corruption.transform(X)

    null_mask = X.isnull() & X_original.notna()

    df_final = X.copy()
    df_final[target] = y

    return df_final, X_original, null_mask 


def compute_imputation_quality(
    X_original: pd.DataFrame,
    X_imputed: pd.DataFrame,
    null_mask: pd.DataFrame,
) -> dict:
    """
    Compara valores imputados com os valores originais apenas
    nas posições onde nulos foram injetados artificialmente.

    Retorna:
      - mae_num  : MAE médio nas colunas numéricas
      - rmse_num : RMSE médio nas colunas numéricas
      - acc_cat  : Acurácia média nas colunas categóricas
      - n_num    : total de posições numéricas avaliadas
      - n_cat    : total de posições categóricas avaliadas
    """
    numeric_cols     = X_original.select_dtypes(include=[np.number]).columns
    categorical_cols = [c for c in X_original.columns if c not in numeric_cols]

    mae_list, rmse_list, acc_list = [], [], []
    n_num, n_cat = 0, 0

    for col in numeric_cols:
        mask = null_mask[col] if col in null_mask.columns else pd.Series(False, index=X_original.index)
        idx  = mask[mask].index
        if len(idx) == 0:
            continue
        orig = pd.to_numeric(X_original.loc[idx, col], errors="coerce").dropna()
        imp  = pd.to_numeric(X_imputed.loc[orig.index, col],  errors="coerce").dropna()
        common = orig.index.intersection(imp.index)
        if len(common) == 0:
            continue
        o, i = orig[common].values, imp[common].values
        mae_list.append(mean_absolute_error(o, i))
        rmse_list.append(np.sqrt(mean_squared_error(o, i)))
        n_num += len(common)

    for col in categorical_cols:
        mask = null_mask[col] if col in null_mask.columns else pd.Series(False, index=X_original.index)
        idx  = mask[mask].index
        if len(idx) == 0:
            continue
        orig = X_original.loc[idx, col].dropna().astype(str)
        imp  = X_imputed.loc[orig.index, col].fillna("__missing__").astype(str)
        common = orig.index.intersection(imp.index)
        if len(common) == 0:
            continue
        acc_list.append(accuracy_score(orig[common], imp[common]))
        n_cat += len(common)

    return {
        "imp_mae_num":  float(np.mean(mae_list))  if mae_list  else None,
        "imp_rmse_num": float(np.mean(rmse_list)) if rmse_list else None,
        "imp_acc_cat":  float(np.mean(acc_list))  if acc_list  else None,
        "imp_n_num":    n_num,
        "imp_n_cat":    n_cat,
    }

def impute_with_miceforest(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    random_state: int = 42,
    iterations: int = 2,
    max_cols: int = 50,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Imputação com MICE + LightGBM via miceforest (método externo ao scikit-learn).

    Configuração:
      - iterations=2: 2 iterações MICE (suficiente para experimentos)
      - data_subset=0.5: usa 50% das linhas para treinar cada modelo interno
        (acelera datasets grandes sem perder qualidade significativa)
      - max_cols=50: datasets com mais colunas usam apenas as top-50 por variância
        (communities_crime tem ~100 cols e fica lento sem essa limitação)
      - Respeita o protocolo: kernel ajustado só no X_train, X_test imputado
        via impute_new_data()

    Limitações documentadas:
      - Colunas categóricas convertidas para códigos numéricos antes do kernel.
      - Em datasets com >50 colunas, apenas as 50 de maior variância são usadas
        para imputação; as demais recebem SimpleImputer(mean) como fallback.
    """
    X_train_n = to_native_dtypes(X_train.copy())
    X_test_n  = to_native_dtypes(X_test.copy())

    # miceforest v6 exige RangeIndex sequencial
    X_train_n = X_train_n.reset_index(drop=True)
    X_test_n  = X_test_n.reset_index(drop=True)

    # Converter categorias para codes numéricos
    cat_cols = X_train_n.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in cat_cols:
        combined = pd.concat([X_train_n[col], X_test_n[col]], axis=0)
        cats = combined.dropna().unique()
        cat_map = {v: i for i, v in enumerate(cats)}
        X_train_n[col] = X_train_n[col].map(cat_map).astype(float)
        X_test_n[col]  = X_test_n[col].map(cat_map).astype(float)

    # Limitar colunas em datasets grandes (evita lentidão no communities_crime)
    all_cols = X_train_n.columns.tolist()
    if len(all_cols) > max_cols:
        # seleciona top-max_cols por variância (colunas mais informativas)
        variances = X_train_n.var(numeric_only=True).nlargest(max_cols)
        selected_cols = variances.index.tolist()
        remaining_cols = [c for c in all_cols if c not in selected_cols]
        print(f"[miceforest] {len(all_cols)} colunas → usando top-{max_cols} por variância")
    else:
        selected_cols  = all_cols
        remaining_cols = []

    X_tr_sel = X_train_n[selected_cols].copy()
    X_te_sel = X_test_n[selected_cols].copy()

    try:
        # API miceforest >= 6.x
        kernel = mf.ImputationKernel(
            X_tr_sel,
            random_state=random_state,
            data_subset=0.5,   # usa 50% das linhas por iteração — acelera sem perder qualidade
        )
        kernel.mice(iterations, verbose=False)

        X_train_imp_sel = kernel.complete_data(dataset=0)
        new_data_kernel = kernel.impute_new_data(
            new_data=X_te_sel,
            random_state=random_state,
        )
        X_test_imp_sel = new_data_kernel.complete_data(dataset=0)

    except Exception as e:
        err_msg = str(e) if str(e).strip() else type(e).__name__
        print(f"  [miceforest] fallback SimpleImputer ({err_msg})")
        from sklearn.impute import SimpleImputer
        imp = SimpleImputer(strategy="mean")
        X_train_imp_sel = pd.DataFrame(
            imp.fit_transform(X_tr_sel),
            columns=X_tr_sel.columns, index=X_tr_sel.index
        )
        X_test_imp_sel = pd.DataFrame(
            imp.transform(X_te_sel),
            columns=X_te_sel.columns, index=X_te_sel.index
        )

    # Reintegrar colunas restantes (imputadas com mean)
    if remaining_cols:
        from sklearn.impute import SimpleImputer
        imp_rem = SimpleImputer(strategy="mean")
        X_tr_rem = pd.DataFrame(
            imp_rem.fit_transform(X_train_n[remaining_cols]),
            columns=remaining_cols, index=X_train_n.index
        )
        X_te_rem = pd.DataFrame(
            imp_rem.transform(X_test_n[remaining_cols]),
            columns=remaining_cols, index=X_test_n.index
        )
        X_train_imp = pd.concat([X_train_imp_sel, X_tr_rem], axis=1)[all_cols]
        X_test_imp  = pd.concat([X_test_imp_sel,  X_te_rem],  axis=1)[all_cols]
    else:
        X_train_imp = X_train_imp_sel
        X_test_imp  = X_test_imp_sel

    return X_train_imp, X_test_imp


def _print_metrics(name: str, metrics: dict, task: str) -> None:
    """Imprime metricas de forma compacta e legivel."""
    if task == "classification":
        acc = metrics.get("accuracy")
        mf1 = metrics.get("macro_f1")
        wf1 = metrics.get("weighted_f1")
        if acc is None:
            print(f"     x  {name:<26} ERRO")
        else:
            bar = "#" * int(wf1 * 20) + "-" * (20 - int(wf1 * 20))
            print(f"     ok {name:<26} acc={acc:.3f}  mf1={mf1:.3f}  wf1={wf1:.3f}  [{bar}]")
    else:
        mae  = metrics.get("mae")
        rmse = metrics.get("rmse")
        r2   = metrics.get("r2")
        if r2 is None:
            print(f"     x  {name:<26} ERRO")
        else:
            bar = "#" * max(0, int(r2 * 20)) + "-" * (20 - max(0, int(r2 * 20)))
            print(f"     ok {name:<26} mae={mae:.4f}  rmse={rmse:.4f}  r2={r2:.4f}  [{bar}]")


def run_classification(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:

    y = pd.Series(y.to_numpy(dtype=object, na_value=np.nan), index=y.index)
    y = y.dropna()
    y = y.astype(str)
    X = X.loc[y.index]
    X = to_native_dtypes(X)  # converte todos ExtensionDtypes para nativos

    class_counts = y.value_counts()
    stratify = y if int(class_counts.min()) >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify
    )

    models = {
        "logistic_regression": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=True)),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", RandomForestClassifier(
                n_estimators=N_ESTIMATORS, 
                random_state=random_state,
                n_jobs=1,
                class_weight="balanced_subsample",
            )),
        ]),
        "lightgbm": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", LGBMClassifier(
                n_estimators=N_ESTIMATORS_LGBM,
                random_state=random_state,
                n_jobs=1,
                class_weight="balanced",
                verbosity=-1,
            )),
        ]),
    }

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = {
            "accuracy": float(accuracy_score(y_test, pred)),
            "macro_f1": float(f1_score(y_test, pred, average="macro")),
            "weighted_f1": float(f1_score(y_test, pred, average="weighted")),
        }

        results[name] = metrics
        _print_metrics(name, metrics, task="classification")

    return results

def run_classification_with_imputation(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float,
    random_state: int,
    X_original: pd.DataFrame = None,
    null_mask: pd.DataFrame = None,
) -> dict:

    IMPUTERS_LIST = [
        "dropna", "simple_mean", "simple_median", "simple_freq",
        "simple_const", "knn_3", "knn_5", "knn_7", "iterative", "miceforest",
    ]

    y = pd.Series(y.to_numpy(dtype=object, na_value=np.nan), index=y.index)
    y = y.dropna()
    y = y.astype(str)
    X = X.loc[y.index].copy()
    X = to_native_dtypes(X)

    if len(y) == 0:
        raise RuntimeError("Target vazia após limpeza.")

    class_counts = y.value_counts(dropna=False)
    stratify = y if int(class_counts.min()) >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=stratify
    )

    results = {}

    for imp in IMPUTERS_LIST:
        print(f"  > imputador: {imp}")

        if imp == "dropna":
            mask_train = ~X_train.isnull().any(axis=1)
            mask_test  = ~X_test.isnull().any(axis=1)
            X_tr = X_train[mask_train]
            y_tr = y_train[mask_train]
            X_te = X_test[mask_test]
            y_te = y_test[mask_test]
            if len(y_tr) == 0 or len(y_te) == 0:
                results[imp] = {}
                continue

        elif imp == "miceforest":
            X_tr_imp, X_te_imp = impute_with_miceforest(
                X_train, X_test, random_state=random_state
            )
            X_tr, y_tr = X_tr_imp, y_train
            X_te, y_te = X_te_imp, y_test

        else:
            X_tr, y_tr, X_te, y_te = X_train, y_train, X_test, y_test

        # ── qualidade da imputação ────────────────────────────────────────────
        imp_quality = {}
        if X_original is not None and null_mask is not None and imp != "dropna":
            try:
                train_idx = X_train.index
                num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
                cat_cols = [c for c in X_train.columns if c not in num_cols]

                if imp == "miceforest":
                    X_tr_for_quality = pd.DataFrame(
                        X_tr.values, index=train_idx, columns=X_train.columns
                    )
                else:
                    # Seleciona o imputador numérico correto diretamente
                    if imp == "simple_mean":
                        num_imp = SimpleImputer(strategy="mean")
                    elif imp == "simple_median":
                        num_imp = SimpleImputer(strategy="median")
                    elif imp == "simple_freq":
                        num_imp = SimpleImputer(strategy="most_frequent")
                    elif imp == "simple_const":
                        num_imp = SimpleImputer(strategy="constant", fill_value=0)
                    elif imp.startswith("knn_"):
                        num_imp = KNNImputer(n_neighbors=int(imp.split("_")[1]))
                    elif imp == "iterative":
                        num_imp = IterativeImputer(
                            max_iter=MAX_ITER_IMPUTER, initial_strategy="mean", random_state=0
                        )
                    else:
                        num_imp = SimpleImputer(strategy="mean")

                    # Imputa numérico
                    if num_cols:
                        arr_num = num_imp.fit_transform(X_tr[num_cols])
                        X_num_imp = pd.DataFrame(arr_num, index=train_idx, columns=num_cols)
                    else:
                        X_num_imp = pd.DataFrame(index=train_idx)

                    # Imputa categórico (sempre most_frequent)
                    if cat_cols:
                        cat_imp = SimpleImputer(strategy="most_frequent")
                        arr_cat = cat_imp.fit_transform(X_tr[cat_cols].astype(object))
                        X_cat_imp = pd.DataFrame(arr_cat, index=train_idx, columns=cat_cols)
                        X_tr_for_quality = pd.concat([X_num_imp, X_cat_imp], axis=1)[X_train.columns]
                    else:
                        X_tr_for_quality = X_num_imp[X_train.columns] if num_cols else X_tr.copy()

                imp_quality = compute_imputation_quality(
                    X_original.loc[train_idx],
                    X_tr_for_quality,
                    null_mask.loc[train_idx],
                )
            except Exception as e:
                print(f"     [imp_quality erro] {imp}: {e}")
                imp_quality = {}
        # ─────────────────────────────────────────────────────────────────────

        models = {
            "logistic_regression": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, True)),
                ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
            ]),
            "random_forest": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, False)),
                ("model", RandomForestClassifier(
                    n_estimators=N_ESTIMATORS, max_depth=10,
                    random_state=random_state, n_jobs=1,
                    class_weight="balanced_subsample",
                )),
            ]),
            "lightgbm": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, False)),
                ("model", LGBMClassifier(
                    n_estimators=N_ESTIMATORS_LGBM, max_depth=10,
                    random_state=random_state, n_jobs=1,
                    class_weight="balanced", verbosity=-1,
                )),
            ]),
        }

        results[imp] = {}

        for name, model in models.items():
            try:
                model.fit(X_tr, y_tr)
                pred = model.predict(X_te)
                metrics = {
                    "accuracy":    float(accuracy_score(y_te, pred)),
                    "macro_f1":    float(f1_score(y_te, pred, average="macro")),
                    "weighted_f1": float(f1_score(y_te, pred, average="weighted")),
                }
            except Exception as e:
                print(f"     x  {name}: {e}")
                metrics = {"accuracy": None, "macro_f1": None, "weighted_f1": None}

            metrics.update(imp_quality)
            results[imp][name] = metrics
            _print_metrics(name, metrics, task="classification")

    return results


def run_regression_with_imputation(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float,
    random_state: int,
    X_original: pd.DataFrame = None,
    null_mask: pd.DataFrame = None,
) -> dict:

    IMPUTERS_LIST = [
        "dropna", "simple_mean", "simple_median", "simple_freq",
        "simple_const", "knn_3", "knn_5", "knn_7", "iterative", "miceforest",
    ]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    results = {}

    for imp in IMPUTERS_LIST:
        print(f"  > imputador: {imp}")

        if imp == "dropna":
            mask_train = ~X_train.isnull().any(axis=1)
            mask_test  = ~X_test.isnull().any(axis=1)
            X_tr = X_train[mask_train]
            y_tr = y_train[mask_train]
            X_te = X_test[mask_test]
            y_te = y_test[mask_test]
            if len(y_tr) == 0 or len(y_te) == 0:
                results[imp] = {}
                continue

        elif imp == "miceforest":
            X_tr_imp, X_te_imp = impute_with_miceforest(
                X_train, X_test, random_state=random_state
            )
            X_tr, y_tr = X_tr_imp, y_train
            X_te, y_te = X_te_imp, y_test

        else:
            X_tr, y_tr, X_te, y_te = X_train, y_train, X_test, y_test

        # ── qualidade da imputação ────────────────────────────────────────────
        imp_quality = {}
        if X_original is not None and null_mask is not None and imp != "dropna":
            try:
                train_idx = X_train.index
                num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
                cat_cols = [c for c in X_train.columns if c not in num_cols]

                if imp == "miceforest":
                    X_tr_for_quality = pd.DataFrame(
                        X_tr.values, index=train_idx, columns=X_train.columns
                    )
                else:
                    if imp == "simple_mean":
                        num_imp = SimpleImputer(strategy="mean")
                    elif imp == "simple_median":
                        num_imp = SimpleImputer(strategy="median")
                    elif imp == "simple_freq":
                        num_imp = SimpleImputer(strategy="most_frequent")
                    elif imp == "simple_const":
                        num_imp = SimpleImputer(strategy="constant", fill_value=0)
                    elif imp.startswith("knn_"):
                        num_imp = KNNImputer(n_neighbors=int(imp.split("_")[1]))
                    elif imp == "iterative":
                        num_imp = IterativeImputer(
                            max_iter=MAX_ITER_IMPUTER, initial_strategy="mean", random_state=0
                        )
                    else:
                        num_imp = SimpleImputer(strategy="mean")

                    if num_cols:
                        arr_num = num_imp.fit_transform(X_tr[num_cols])
                        X_num_imp = pd.DataFrame(arr_num, index=train_idx, columns=num_cols)
                    else:
                        X_num_imp = pd.DataFrame(index=train_idx)

                    if cat_cols:
                        cat_imp = SimpleImputer(strategy="most_frequent")
                        arr_cat = cat_imp.fit_transform(X_tr[cat_cols].astype(object))
                        X_cat_imp = pd.DataFrame(arr_cat, index=train_idx, columns=cat_cols)
                        X_tr_for_quality = pd.concat([X_num_imp, X_cat_imp], axis=1)[X_train.columns]
                    else:
                        X_tr_for_quality = X_num_imp[X_train.columns] if num_cols else X_tr.copy()

                imp_quality = compute_imputation_quality(
                    X_original.loc[train_idx],
                    X_tr_for_quality,
                    null_mask.loc[train_idx],
                )
            except Exception as e:
                print(f"     [imp_quality erro] {imp}: {e}")
                imp_quality = {}
        # ─────────────────────────────────────────────────────────────────────

        models = {
            "ridge_regression": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, True)),
                ("model", Ridge(alpha=1.0)),
            ]),
            "random_forest": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, False)),
                ("model", RandomForestRegressor(
                    n_estimators=N_ESTIMATORS, max_depth=10,
                    random_state=random_state, n_jobs=1,
                )),
            ]),
            "lightgbm": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, False)),
                ("model", LGBMRegressor(
                    n_estimators=N_ESTIMATORS_LGBM, max_depth=10,
                    random_state=random_state, n_jobs=1, verbosity=-1,
                )),
            ]),
        }

        results[imp] = {}

        for name, model in models.items():
            try:
                model.fit(X_tr, y_tr)
                pred = model.predict(X_te)
                mse = mean_squared_error(y_te, pred)
                metrics = {
                    "mae":  float(mean_absolute_error(y_te, pred)),
                    "rmse": float(np.sqrt(mse)),
                    "r2":   float(r2_score(y_te, pred)),
                }
            except Exception as e:
                print(f"     x  {name}: {e}")
                metrics = {"mae": None, "rmse": None, "r2": None}

            metrics.update(imp_quality)
            results[imp][name] = metrics
            _print_metrics(name, metrics, task="regression")

    return results

def run_regression(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    models = {
        "ridge_regression": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=True)),
            ("model", Ridge(alpha=1.0)),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", RandomForestRegressor(
                n_estimators=N_ESTIMATORS, 
                max_depth=10,
                random_state=random_state,
                n_jobs=1
            )),
        ]),
        "lightgbm": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", LGBMRegressor(
                n_estimators=N_ESTIMATORS_LGBM,
                max_depth=10,
                random_state=random_state,
                n_jobs=1,
                verbosity=-1,
            )),
        ]),
    }

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        mse = mean_squared_error(y_test, pred)
        metrics = {
            "mae": float(mean_absolute_error(y_test, pred)),
            "rmse": float(np.sqrt(mse)),
            "r2": float(r2_score(y_test, pred)),
        }

        results[name] = metrics
        _print_metrics(name, metrics, task="regression")

    return results

# FUNÇÃO PRINCIPAL
import time
import sys
import json

def run_one_dataset(
    key: str,
    out_dir: str = "data",
    test_size: float = 0.2,
    random_state: int = 42
) -> dict:

    data_dir = Path(out_dir)
    df, target, task = load_dataset(key, data_dir=data_dir)
    
    if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
        df = df.sample(n=SAMPLE_SIZE, random_state=random_state).reset_index(drop=True)
        print(f"  [FAST] amostra de {SAMPLE_SIZE} linhas")

    SEP = "=" * 72
    task_icon = "CLASSIFICACAO" if task == "classification" else "REGRESSAO"
    print(f"\n{SEP}")
    print(f"  {task_icon}  |  {key.upper()}  |  target: {target}")
    print(SEP)

    df = normalize_missing_values(df)
    df = to_native_dtypes(df) 

    df[target] = df[target].astype(object)
    df = df[df[target].notna()].copy()

    if df.empty:
        raise RuntimeError("Dataset vazio após limpeza.")

    X, y = split_X_y(df, target)

    y = pd.Series([np.nan if v is pd.NA else v for v in y.astype(object)], index=y.index, dtype=object)

    if task == "classification":
        y = y.dropna()
        X = X.loc[y.index]

        X = to_native_dtypes(X)

        y = y.astype(object).astype(str)

        vc = y.value_counts()
        total = vc.sum()
        dist_str = "  |  ".join(f"{cls}: {cnt} ({cnt/total:.0%})" for cls, cnt in vc.items())
        print(f"  classes: {dist_str}")

        baseline_results = run_classification(X, y, test_size, random_state)

    else:
        y = pd.to_numeric(y, errors="coerce")
        valid = y.notna()

        X = X.loc[valid]
        # NOVO: converte X também para dtypes nativos
        X = to_native_dtypes(X)
        y = y.loc[valid]

        baseline_results = run_regression(X, y, test_size, random_state)

    # EXPERIMENTOS
    missing_mechanisms = MISSING_MECHANISMS 
    missing_rates      = MISSING_RATES    

    all_results = []

    all_results.append({
        "dataset": key,
        "mechanism": "NONE",
        "rate": 0,
        "results": baseline_results
    })

    for mech in missing_mechanisms:
        for rate in missing_rates:
            pct = int(rate * 100)
            print(f"\n  EXPERIMENTO  {mech}  {pct:2d}%  [{key}]")
            print(f"  {chr(45)*50}")

            df_missing, X_original_exp, null_mask_exp = inject_missing_values(
                df,
                target=target,
                mechanism=mech,
                missing_rate=rate,
                random_state=random_state
            )

            X, y = split_X_y(df_missing, target)


            X = to_native_dtypes(X)
            y = pd.Series(
                [np.nan if (v is pd.NA or (isinstance(v, float) and np.isnan(v))) else v 
                for v in y],
                index=y.index,
                dtype=object
            )

            if task == "classification":
                valid = y.notna()
                X = X.loc[valid]
                y = y.loc[valid].astype(str)
                res = run_classification_with_imputation(X, y, test_size, random_state,
                                         X_original_exp, null_mask_exp)
            else:
                y = pd.to_numeric(y, errors="coerce")
                valid = y.notna()
                X = X.loc[valid]
                y = y.loc[valid]
                res = run_regression_with_imputation(X, y, test_size, random_state,
                                     X_original_exp, null_mask_exp)

            all_results.append({
                "dataset": key,
                "mechanism": mech,
                "rate": rate,
                "results": res
            })

    return {
        "key": key,
        "task": task,
        "results": baseline_results,
        "experiments": all_results
    }

In [14]:
# 5) Configuracao da execucao

# ============================================================
# 🚀 FAST_MODE: True = rápido para testes  (~2-5 min)
#               False = completo para resultados finais (~120 min)
# ============================================================
FAST_MODE = False

# Parametros globais
out_dir = "data"
test_size = 0.2
random_state = 42

datasets_experiment = [
    "adult",
    "bank_marketing",
    "air_quality_uci",
    "communities_crime",
]

if FAST_MODE:
    MISSING_MECHANISMS = ["MCAR"]
    MISSING_RATES      = [0.05]
    N_ESTIMATORS       = 10
    N_ESTIMATORS_LGBM  = 50   # mínimo razoável para LightGBM no FAST_MODE
    MAX_ITER_IMPUTER   = 2
    SAMPLE_SIZE        = 3000
    print("⚡ FAST MODE — resultados aproximados para testes")
else:
    MISSING_MECHANISMS = ["MCAR", "MAR", "MNAR"]
    MISSING_RATES      = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
    N_ESTIMATORS       = 50
    N_ESTIMATORS_LGBM  = 100  # LightGBM leaf-wise precisa de mais estimadores
    MAX_ITER_IMPUTER   = 5
    SAMPLE_SIZE        = None
    print("🔬 FULL MODE — execução completa")

datasets_experiment

🔬 FULL MODE — execução completa


['adult', 'bank_marketing', 'air_quality_uci', 'communities_crime']

In [15]:
#barra de progresso
from tqdm import tqdm
from joblib import Parallel, delayed

class ParallelTqdm(Parallel):
    def __init__(self, total=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.total = total

    def __call__(self, *args, **kwargs):
        with tqdm(total=self.total) as self._pbar:
            return Parallel.__call__(self, *args, **kwargs)

    def print_progress(self):
        if self._pbar is not None:
            self._pbar.n = self.n_completed_tasks
            self._pbar.refresh()

In [16]:
from joblib import Parallel, delayed
import multiprocessing
from pathlib import Path
import pandas as pd
import numpy as np

if __name__ == "__main__":

    N_CORES     = multiprocessing.cpu_count()
    N_JOBS_OUTER = min(len(datasets_experiment), N_CORES // 2)

    print(f"Total cores: {N_CORES}")
    print(f"Paralelismo externo (datasets): {N_JOBS_OUTER}")

    # ── Sementes para repetições ──────────────────────────────────────────────
    RANDOM_SEEDS = [42, 123, 456]

    # ── Função segura por (dataset, semente) ─────────────────────────────────
    def run_dataset_seed_safe(key, seed):
        try:
            return run_one_dataset(
                key=key,
                out_dir=out_dir,
                test_size=test_size,
                random_state=seed,
            )
        except Exception as e:
            print(f"\n[ERRO] key={key} seed={seed}: {e}")
            return {
                "key": key,
                "task": DATASETS[key]["task"],
                "seed": seed,
                "error": str(e),
            }

    # ── Combinações dataset × semente ────────────────────────────────────────
    combos = [(key, seed) for key in datasets_experiment for seed in RANDOM_SEEDS]

    # ── Execução paralela ─────────────────────────────────────────────────────
    all_runs = ParallelTqdm(
        n_jobs=N_JOBS_OUTER,
        total=len(combos),
        backend="threading",
        prefer="threads",
    )(
        delayed(run_dataset_seed_safe)(key, seed) for key, seed in combos
    )

    # ── Montar CSV detalhado com TODOS os cenários ────────────────────────────
    detail_rows = []

    for run in all_runs:
        if "error" in run and "experiments" not in run:
            continue  # pula runs com erro total

        key  = run["key"]
        task = run["task"]
        seed = run.get("seed", run.get("random_state", 42))

        # Baseline (mechanism=NONE, rate=0, imputer=baseline)
        for model, metrics in run.get("results", {}).items():
            if metrics is None:
                continue
            row = {
                "dataset":   key,
                "task":      task,
                "seed":      seed,
                "mechanism": "NONE",
                "rate":      0.0,
                "imputer":   "baseline_dropna",
                "model":     model,
            }
            row.update(metrics)
            detail_rows.append(row)

        # Experimentos com nulos injetados
        for exp in run.get("experiments", []):
            if exp.get("mechanism") == "NONE":
                continue  # já capturado acima
            for imputer, imp_results in exp.get("results", {}).items():
                for model, metrics in imp_results.items():
                    if metrics is None:
                        continue
                    row = {
                        "dataset":   key,
                        "task":      task,
                        "seed":      seed,
                        "mechanism": exp["mechanism"],
                        "rate":      exp["rate"],
                        "imputer":   imputer,
                        "model":     model,
                    }
                    row.update(metrics)
                    detail_rows.append(row)

    detail_df = pd.DataFrame(detail_rows)
    data_path  = Path(out_dir)
    data_path.mkdir(parents=True, exist_ok=True)

    detail_path = data_path / "all_results.csv"
    detail_df.to_csv(detail_path, index=False)
    print(f"\nCSV detalhado salvo: {detail_path}  ({len(detail_df)} linhas)")

    # ── Resumo com média ± desvio padrão ─────────────────────────────────────
    grp_cols = ["dataset", "task", "mechanism", "rate", "imputer", "model"]

    # Métricas disponíveis dependem da task
    clf_metrics = ["accuracy", "macro_f1", "weighted_f1"]
    reg_metrics = ["mae", "rmse", "r2"]
    all_metrics = clf_metrics + reg_metrics

    existing_metrics = [c for c in all_metrics if c in detail_df.columns]

    summary = (
        detail_df
        .groupby(grp_cols)[existing_metrics]
        .agg(["mean", "std"])
        .round(4)
    )
    summary_path = data_path / "summary_results.csv"
    summary.to_csv(summary_path)
    print(f"CSV resumo (média±std) salvo: {summary_path}")

    # ── Tabelas de baseline (melhor modelo, média das sementes) ───────────────
    baseline_df = detail_df[detail_df["mechanism"] == "NONE"].copy()

    if "accuracy" in baseline_df.columns:
        clf_base = (
            baseline_df[baseline_df["task"] == "classification"]
            .groupby(["dataset", "model"])[clf_metrics]
            .mean()
            .round(4)
            .reset_index()
        )
        if not clf_base.empty:
            best_clf = clf_base.loc[
                clf_base.groupby("dataset")["weighted_f1"].idxmax()
            ]
            print("\n=== Classificacao (baseline, média 3 sementes) ===")
            display(best_clf)
            best_clf.to_csv(data_path / "classification_summary.csv", index=False)
            print("Salvo em:", data_path / "classification_summary.csv")

    if "r2" in baseline_df.columns:
        reg_base = (
            baseline_df[baseline_df["task"] == "regression"]
            .groupby(["dataset", "model"])[reg_metrics]
            .mean()
            .round(4)
            .reset_index()
        )
        if not reg_base.empty:
            best_reg = reg_base.loc[
                reg_base.groupby("dataset")["r2"].idxmax()
            ]
            print("\n=== Regressao (baseline, média 3 sementes) ===")
            display(best_reg)
            best_reg.to_csv(data_path / "regression_summary.csv", index=False)
            print("Salvo em:", data_path / "regression_summary.csv")


Total cores: 12
Paralelismo externo (datasets): 4


  0%|          | 0/12 [00:00<?, ?it/s]

[cache] data\bank_marketing\bank.zip[cache] data\adult\adult.data
[cache] data\adult\adult.data
[cache] data\adult\adult.data

[cache] data\bank_marketing\extracted
[cache] data\adult\adult.test
[cache] data\adult\adult.test
[cache] data\adult\adult.test

  CLASSIFICACAO  |  BANK_MARKETING  |  target: y

  CLASSIFICACAO  |  ADULT  |  target: income

  CLASSIFICACAO  |  ADULT  |  target: income

  CLASSIFICACAO  |  ADULT  |  target: income
  classes: no: 39922 (88%)  |  yes: 5289 (12%)
  classes: <=50K: 37155 (76%)  |  >50K: 11687 (24%)
  classes: <=50K: 37155 (76%)  |  >50K: 11687 (24%)
  classes: <=50K: 37155 (76%)  |  >50K: 11687 (24%)
     ok logistic_regression        acc=0.846  mf1=0.730  wf1=0.865  [#################---]
     ok logistic_regression        acc=0.816  mf1=0.780  wf1=0.827  [################----]
     ok logistic_regression        acc=0.808  mf1=0.769  wf1=0.819  [################----]
     ok logistic_regression        acc=0.808  mf1=0.770  wf1=0.819  [############

  8%|▊         | 1/12 [7:25:12<81:37:12, 26712.09s/it]

     ok lightgbm                   acc=0.851  mf1=0.746  wf1=0.871  [#################---]
[cache] data\bank_marketing\bank.zip
[cache] data\bank_marketing\extracted

  CLASSIFICACAO  |  BANK_MARKETING  |  target: y
  classes: no: 39922 (88%)  |  yes: 5289 (12%)
     ok logistic_regression        acc=0.833  mf1=0.713  wf1=0.855  [#################---]
     ok random_forest              acc=0.904  mf1=0.697  wf1=0.889  [#################---]
     ok lightgbm                   acc=0.859  mf1=0.753  wf1=0.877  [#################---]

  EXPERIMENTO  MCAR   1%  [bank_marketing]
  --------------------------------------------------
  > imputador: dropna
     ok logistic_regression        acc=0.811  mf1=0.772  wf1=0.821  [################----]
     ok random_forest              acc=0.826  mf1=0.779  wf1=0.832  [################----]
     ok lightgbm                   acc=0.827  mf1=0.785  wf1=0.834  [################----]
  > imputador: simple_mean
     ok logistic_regression        acc=0.834 

...

     ok lightgbm                   mae=0.0933  rmse=0.1396  r2=0.6371  [############--------]
  > imputador: miceforest
[miceforest] 122 colunas → usando top-50 por variância
  [miceforest] fallback SimpleImputer (AssertionError)

[ERRO] key=communities_crime seed=456: Shape of passed values is (1595, 71), indices imply (1595, 72)

CSV detalhado salvo: data\all_results.csv  (5574 linhas)
CSV resumo (média±std) salvo: data\summary_results.csv

=== Classificacao (baseline, média 3 sementes) ===


,dataset,model,accuracy,macro_f1,weighted_f1
2,adult,random_forest,0.8551,0.7875,0.8500
5,bank_marketing,random_forest,0.9037,0.6989,0.8891


Salvo em: data\classification_summary.csv

=== Regressao (baseline, média 3 sementes) ===


,dataset,model,mae,rmse,r2
1,air_quality_uci,random_forest,0.0157,0.1150,0.9998
3,communities_crime,lightgbm,0.0912,0.1366,0.6300


Salvo em: data\regression_summary.csv
